# Does perp flow and positioning improve 15-minute BTC spot prediction?

Analysis notebook: it only calls functions from `src/` and runs top to bottom on a fresh kernel.

**Before running:** build the data once from the repo root with `python -m src.pipeline` (see the README). Figures are saved to `report/figures` and tables to `report/tables`.

Numbers quoted in the text refer to BTCUSDT, 1 January – 30 June 2026.

**H1:** adding perp flow and positioning features (block B) to spot-only features (block A) improves out-of-sample 15-minute spot prediction. **H2:** the perp leads spot.

In [ ]:
import pandas as pd
from IPython.display import Image, display

from src import descriptive, plots
from src.config import load_config
from src.features import BLOCK_A, BLOCK_B, build_features, sensitivity_table
from src.panel import build_panel

cfg = load_config()
symbol = cfg["symbol"]
panel = build_panel(cfg, symbol, cfg["start_date"], cfg["end_date"])
print(f"{symbol}: {len(panel):,} minutes, {cfg['start_date']} to {cfg['end_date']}")

## 1. Data quality and market structure (blocks A–B)

In [ ]:
display(Image(plots.figure_1(descriptive.missing_minutes(panel), cfg)))

**Figure 1.** No minute is missing on either market over the window: the panel is complete, so no feature window is lost to gaps.

In [ ]:
tracking = descriptive.tracking_stats(panel)
descriptive.save_table(tracking, f"tracking_{symbol}", cfg)
tracking.round(2)

Perp and spot closes differ by a median 4.9 bps (99th percentile 7.0 bps, maximum 17 bps), so the join and the units are correct. The mean gap is −4.9 bps: the perp trades slightly below spot over the whole window.

In [ ]:
display(Image(plots.figure_2(descriptive.volume_share(panel), cfg)))

**Figure 2.** The perp carries 85–92% of the traded value every day (median 90%). Most BTC trading on Binance happens on the perp, which is why we look there for information that spot has not absorbed yet (H1, H2).

In [ ]:
display(Image(plots.figure_3(descriptive.intraday_profile(panel), cfg)))

**Figure 3.** Volume and volatility both peak at 14–15 UTC (US open and data releases) and are lowest around 04–10 UTC, on both markets. This intraday cycle is the reason for the time-of-day features in block A.

## 2. Returns and volatility (block C)

In [ ]:
samples = descriptive.return_samples(panel, cfg["horizon_minutes"])
moments = descriptive.return_moments(samples)
descriptive.save_table(moments, f"return_moments_{symbol}", cfg)
moments.round(2)

In [ ]:
display(Image(plots.figure_4(samples, cfg)))

**Figure 4.** Returns are strongly fat-tailed: excess kurtosis is 20.4 at 1 minute and 16.9 at 15 minutes (0 for a normal distribution), with 1-minute moves of up to 167 bps. The histograms are more peaked than the normal curve and the QQ plots bend away at both ends. A few large moves would dominate a squared-error fit, which supports the three-class target with a volatility-scaled threshold.

In [ ]:
display(Image(plots.figure_5(descriptive.acf_table(panel), cfg)))

**Figure 5.** Returns are uncorrelated from lag 2 on, apart from small negative values at lags 2–4. The lag-1 value of 0.21 is mechanical: averaging prices within each minute (VWAP) splits a move across two consecutive returns (Working, 1960). It also explains why the 15-minute std (25.9 bps) is above 5.8 × √15 ≈ 22.4 bps. Absolute returns stay strongly autocorrelated for an hour (0.38 at lag 1, still above 0.2 at lag 60): volatility clusters, which motivates the realised-volatility features. Part of this persistence comes from the intraday cycle of Figure 3.

In [ ]:
stationarity = descriptive.stationarity_tests(descriptive.five_minute_series(panel))
descriptive.save_table(stationarity, f"stationarity_{symbol}", cfg)
stationarity.round(4)

ADF rejects a unit root for all four series (5-minute bars). KPSS agrees for spot returns and perp flow, but flags the basis, whose mean drifts slowly, and, marginally, spot flow. All four can be used as features without differencing.

## 3. Perp-specific signals (block D)

In [ ]:
sensitivity_table(panel)

The deleveraging proxy (Binance publishes no complete history of USD-M liquidations) reproduces the spec's table: at the 1% open interest threshold, 161 suspected deleveraging events (98 with a falling price, 63 with a rising price) in 127 clusters.

In [ ]:
half_life = descriptive.basis_half_life(panel)
descriptive.save_table(half_life, f"basis_half_life_{symbol}", cfg)
clusters = descriptive.deleveraging_clusters(panel)
display(Image(plots.figure_6(descriptive.basis_over_time(panel), clusters, half_life, cfg)))
half_life.round(3)

**Figure 6.** The basis averages −4.9 bps. Minute-level deviations disappear within about a minute (AR(1) φ = 0.36, half-life 0.7 minutes), while the average level drifts slowly over weeks (KPSS above). The suspected deleveraging clusters are densest in early February.

In [ ]:
display(Image(plots.figure_7(descriptive.leverage_series(panel), cfg)))

**Figure 7.** Open interest falls from about 9 to 5.5 billion USDT in early February while funding turns negative until May. Part of the fall is the lower BTC price, since open interest is shown in USDT, but it coincides with the densest group of deleveraging clusters in Figure 6 (which measure open interest in BTC) and with the early-February 2026 sell-off, when BTC fell to about $60,000 on 6 February amid a broad unwind of leverage. Open interest rebuilds from March to mid-May and drops again in early June. The basis stays negative even when funding is positive: Binance keeps funding at its 0.01% default while its own premium index, measured against a multi-exchange spot index, stays between −0.04% and +0.06%..

In [ ]:
flows = descriptive.flow_correlation(panel, [1, 5, 15, 60])
descriptive.save_table(flows, f"flow_correlation_{symbol}", cfg)
flows.round(3)

Spot and perp signed flows correlate at about 0.5 (0.52 over 1 minute, 0.48 over 60 minutes): perp flow is far from a copy of spot flow, so it can carry information that block A lacks. Whether that extra part helps predict spot is what the models test.

In [ ]:
correlation = build_features(panel, cfg)[BLOCK_A + BLOCK_B].corr()
display(Image(plots.figure_10(correlation, cfg)))
redundant = descriptive.redundant_pairs(correlation)
descriptive.save_table(redundant, f"redundant_features_{symbol}", cfg)
redundant.round(3)

**Figure 10.** Within block A, the realised-volatility features are highly correlated (0.8–0.9). `b_basis_chg_L` duplicates `b_lead_L` (correlation 1.000), and `b_flow_div_L` is exactly `b_flow_L − a_flow_L`, a combination the pairwise check cannot flag. The models drop both kinds before fitting.

## 4. Lead-lag (block E)

In [ ]:
display(Image(plots.figure_8(descriptive.lead_lag_table(panel), cfg)))

**Figure 8.** The correlation of perp return, perp flow share and basis with the next minute's spot return is 0.25, 0.25 and 0.17, against 0.19, 0.05 and 0.07 with spot moving first; from two minutes on, every correlation is indistinguishable from zero. The grey band (±1.96/√N) assumes independent data and is too narrow, since returns are autocorrelated. Part of the one-minute correlation is mechanical (VWAP averaging, Figure 5); the asymmetry between +1 and −1 is the evidence of a lead.

In [ ]:
granger = descriptive.granger(panel)
descriptive.save_table(granger, f"granger_{symbol}", cfg)
granger.round(4)

With about 260,000 minutes every Granger test is significant, so the size of the effect matters: the perp's past adds 3.1–3.6% of explained variance to the next spot return (`delta_r2`), against at most 0.6% in the other direction. Granger causality is predictive, not causal.

In [ ]:
placebo = descriptive.placebo_events(panel, clusters)
study = descriptive.event_study(panel, clusters)
reference = descriptive.event_study(panel, placebo)
descriptive.save_table(study, f"event_study_{symbol}", cfg)
display(Image(plots.figure_9(study, reference, cfg)))

**Figure 9.** After the event bar, spot keeps moving in the direction of the flush for about 5 minutes and then stays flat: −27 bps after suspected long deleveraging and +18 bps after suspected short deleveraging, while equally large perp moves without an open interest drop slightly reverse. The basis barely moves (less than 1 bps). Caveats: a cluster can contain further deleveraging bars after time 0, windows of nearby clusters can overlap, and the placebo matches the size of the move but not the trend before it.

In [ ]:
# Appendix: the same event study at the 0.5% and 2% open interest thresholds
for oi_q, suffix in ((0.005, "_q0.5"), (0.02, "_q2")):
    appendix_clusters = descriptive.deleveraging_clusters(panel, oi_q)
    appendix_placebo = descriptive.placebo_events(panel, appendix_clusters)
    display(Image(plots.figure_9(descriptive.event_study(panel, appendix_clusters),
                                 descriptive.event_study(panel, appendix_placebo), cfg, suffix)))

**Appendix.** The direction is the same at both thresholds: stronger with the stricter 0.5% cut-off (−28 to −40 bps after long deleveraging; the 27 short clusters are too few for a clear band) and weaker at 2% (−15 and +10 bps), as expected if larger flushes matter more.

## 5. Descriptive verdict

The perp leads spot, but only by about one minute. The correlation of perp return, perp flow share and basis with the next minute's spot return is 0.25, 0.25 and 0.17, against 0.19, 0.05 and 0.07 in the other direction, and is indistinguishable from zero from two minutes on; the Granger tests agree (the perp's past adds 3.1–3.6% of explained variance to spot, at most 0.6% the other way), though part of the one-minute correlation is mechanical, from averaging prices within each minute. Around the 127 suspected deleveraging clusters, spot keeps moving in the direction of the flush for about five minutes after the event bar (−27 bps after long and +18 bps after short deleveraging), while equally large moves without an open interest drop do not continue, and this holds at the 0.5% and 2% thresholds. The lead therefore fades well within the 15-minute horizon. Since features only use minutes up to t − 1, the target starts at t, and deleveraging bars are only known five minutes after they close, we expect block B to add little or nothing over block A at 15 minutes: a weak or null result for H1 would be consistent with this evidence.

## 6. Models: block A vs A + B (to be completed)

Section for the ablation results (spec section 8): the results table across walk-forward folds, Figure 11 (forecast gain of A + B over A per fold, with Clark-West p-values) and Figure 12 (coefficients and permutation importance of the perp features).